In [ ]:
# NYC taxi: neural net, trained in its own kernel
# Run NYC-TAXI-FINAL.ipynb first: its train/test split cell saves model_data/*.parquet.
# A fresh kernel holds only the split and this small network, so it can't run out of
# memory the way the big notebook did, and XGBoost's OpenMP library is never loaded.
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"   # harmless here; protects against libomp clashes

import gc
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

DATA_DIR = Path("model_data")
MODEL_DIR = Path("saved_models")
MODEL_DIR.mkdir(exist_ok=True)

CAT_COLS = ["VendorID", "PULocationID", "DOLocationID"]
NUM_COLS = ["pickup_hour", "pickup_month", "pickup_day", "trip_distance", "duration_min"]

X_train = pd.read_parquet(DATA_DIR / "X_train.parquet")
X_test = pd.read_parquet(DATA_DIR / "X_test.parquet")
y_train = pd.read_parquet(DATA_DIR / "y_train.parquet").squeeze("columns")
y_test = pd.read_parquet(DATA_DIR / "y_test.parquet").squeeze("columns")
print(f"Loaded {len(X_train):,} train / {len(X_test):,} test rows")

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def subsample(X, y, n, seed=42):
    """Same sampler as the main notebook, so the NN sees the same rows as before."""
    if n >= len(X):
        return X, y
    idx = np.sort(np.random.default_rng(seed).choice(len(X), size=n, replace=False))
    return X.iloc[idx], y.iloc[idx]

# Shared tuning set: identical to the main notebook's (same sampler, seed and split),
# so the neural net is tuned on exactly the same rows as every other model.
TUNE_ROWS = 500_000
SELECT_BY = "RMSE"
tune_X, tune_y = subsample(X_train, y_train, TUNE_ROWS, seed=7)
tune_Xtr, tune_Xval, tune_ytr, tune_yval = train_test_split(
    tune_X, tune_y, test_size=0.2, random_state=42
)
del tune_X, tune_y
gc.collect()
print(f"Tuning set: {len(tune_Xtr):,} fit rows / {len(tune_Xval):,} validation rows")

In [ ]:
# %% Neural net with embeddings for zones/vendor: tuning + final training
# Hyperparameters tuned (random search: 8 combinations from the grid below):
#   - lr: Adam learning rate (step size).
#   - weight_decay: penalty that keeps weights small (AdamW's decoupled version).
#   - emb_dim: length of the learned vector for each pickup/dropoff zone.
#   - hidden: layer widths, e.g. "128x64" = two hidden layers of 128 and 64 units.
#   - dropout: share of units randomly switched off in training (regularization).
# Each candidate trains for 4 epochs on the shared 400k tuning rows and is scored on
# the 100k validation rows. A short budget ranks settings reliably at a fraction of
# the cost; the winner then trains for the full 15 epochs on 2M rows.
# Number of epochs is not searched: the final run keeps the epoch with the best
# validation score (early stopping), which picks it automatically.
# Crash/RAM safety: runs on CPU (Apple's MPS backend can hard-crash Python), 4
# threads, embedding sizes taken from the data, one network in memory at a time.
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
import time

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import ParameterSampler

torch.set_num_threads(4)

USE_ACCELERATOR = False   # set True to try CUDA/MPS once the CPU run works
device = ("cuda" if USE_ACCELERATOR and torch.cuda.is_available()
          else "mps" if USE_ACCELERATOR and torch.backends.mps.is_available()
          else "cpu")
print("Training on:", device)

NN_TRAIN_ROWS = 2_000_000
EPOCHS, BATCH = 15, 4096

NN_SPACE = {
    "lr": [3e-4, 1e-3, 3e-3],
    "weight_decay": [0.0, 1e-4, 1e-3, 1e-2],
    "emb_dim": [8, 16, 32],
    "hidden": ["64x32", "128x64", "256x128"],
    "dropout": [0.0, 0.1],
}
NN_N_ITER = 8
NN_TUNE_EPOCHS = 4

# Embedding table sizes: largest ID seen anywhere (train or test) + 1
CARD = {c: int(max(X_train[c].max(), X_test[c].max())) + 1 for c in CAT_COLS}
print("Embedding table sizes:", CARD)

num_mean, num_std = X_train[NUM_COLS].mean(), X_train[NUM_COLS].std()
y_mean, y_std = float(y_train.mean()), float(y_train.std())

def to_tensors(X):
    cat = torch.tensor(X[CAT_COLS].to_numpy(dtype="int64"))
    num = torch.tensor(((X[NUM_COLS] - num_mean) / num_std).to_numpy(dtype="float32"))
    return cat, num

class FareNet(nn.Module):
    def __init__(self, n_num, card, emb_dim=16, hidden="128x64", dropout=0.0):
        super().__init__()
        self.veh = nn.Embedding(card["VendorID"], 2)
        self.pu = nn.Embedding(card["PULocationID"], emb_dim)
        self.do = nn.Embedding(card["DOLocationID"], emb_dim)
        layers, d_in = [], 2 + 2 * emb_dim + n_num
        for width in (int(w) for w in hidden.split("x")):
            layers += [nn.Linear(d_in, width), nn.ReLU()]
            if dropout:
                layers.append(nn.Dropout(dropout))
            d_in = width
        layers.append(nn.Linear(d_in, 1))
        self.mlp = nn.Sequential(*layers)
    def forward(self, cat, num):
        z = torch.cat([self.veh(cat[:, 0]), self.pu(cat[:, 1]), self.do(cat[:, 2]), num], dim=1)
        return self.mlp(z).squeeze(-1)

def make_loader(X, y, shuffle):
    cat, num = to_tensors(X)
    tgt = torch.tensor(((y - y_mean) / y_std).to_numpy(dtype="float32"))
    return DataLoader(TensorDataset(cat, num, tgt), batch_size=BATCH, shuffle=shuffle)

def train_net(cfg, train_loader, val_loader, epochs, verbose=True):
    """Train one network; return it at its best validation epoch, plus that RMSE ($)."""
    torch.manual_seed(42)
    net = FareNet(len(NUM_COLS), CARD, cfg["emb_dim"], cfg["hidden"], cfg["dropout"]).to(device)
    optimizer = torch.optim.AdamW(net.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=1)
    loss_fn = nn.MSELoss()
    best_rmse, best_state = float("inf"), None
    for epoch in range(epochs):
        net.train()
        for cat, num, tgt in train_loader:
            cat, num, tgt = cat.to(device), num.to(device), tgt.to(device)
            optimizer.zero_grad()
            loss_fn(net(cat, num), tgt).backward()
            optimizer.step()
        net.eval()
        sq_err, n = 0.0, 0
        with torch.no_grad():
            for cat, num, tgt in val_loader:
                pred = net(cat.to(device), num.to(device)).cpu()
                sq_err += ((pred - tgt) ** 2).sum().item()
                n += len(tgt)
        val_rmse = (sq_err / n) ** 0.5 * y_std      # back to dollars
        scheduler.step(val_rmse)
        if val_rmse < best_rmse:
            best_rmse = val_rmse
            best_state = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}
        if verbose:
            print(f"Epoch {epoch + 1:2d}  val RMSE ${val_rmse:.4f}")
    net.load_state_dict(best_state)
    net.eval()
    return net, best_rmse

def predict_with(net, X, chunk=262_144):
    out = []
    with torch.no_grad():
        for i in range(0, len(X), chunk):
            cat, num = to_tensors(X.iloc[i:i + chunk])
            out.append(net(cat.to(device), num.to(device)).cpu().numpy())
    return np.concatenate(out) * y_std + y_mean

# ---- 1. Random search on the shared tuning rows
tune_train_loader = make_loader(tune_Xtr, tune_ytr, True)
tune_val_loader = make_loader(tune_Xval, tune_yval, False)
nn_configs = list(ParameterSampler(NN_SPACE, n_iter=NN_N_ITER, random_state=42))
rows = []
for i, cfg in enumerate(nn_configs, 1):
    t0 = time.time()
    cand, _ = train_net(cfg, tune_train_loader, tune_val_loader, NN_TUNE_EPOCHS, verbose=False)
    pred = predict_with(cand, tune_Xval)
    rows.append({**cfg,
                 "val_RMSE": float(rmse(tune_yval, pred)),
                 "val_MAE": float(mean_absolute_error(tune_yval, pred)),
                 "secs": round(time.time() - t0, 1)})
    print(f"  [NeuralNet {i}/{len(nn_configs)}] {cfg} -> "
          f"val RMSE {rows[-1]['val_RMSE']:.4f} | val MAE {rows[-1]['val_MAE']:.4f}")
    del cand, pred
    gc.collect()
del tune_train_loader, tune_val_loader

nn_tuning = pd.DataFrame(rows)
best_cfg = nn_configs[int(nn_tuning[f"val_{SELECT_BY}"].idxmin())]
nn_tuning = nn_tuning.sort_values(f"val_{SELECT_BY}").reset_index(drop=True)
nn_tuning.to_parquet(MODEL_DIR / "nn_tuning.parquet")   # shown in the main notebook's summary
print("\nTop settings by validation", SELECT_BY)
print(nn_tuning.head(5).round(4).to_string(index=False))
print("\nBest:", best_cfg)

# ---- 2. Final training: best settings, 2M rows, full epochs
nn_X, nn_y = subsample(X_train, y_train, NN_TRAIN_ROWS)
nn_Xtr, nn_Xval, nn_ytr, nn_yval = train_test_split(nn_X, nn_y, test_size=0.05, random_state=42)
train_loader = make_loader(nn_Xtr, nn_ytr, True)
val_loader = make_loader(nn_Xval, nn_yval, False)
del nn_X, nn_y, nn_Xtr, nn_Xval, nn_ytr, nn_yval   # loaders hold their own copies

net, best_rmse = train_net(best_cfg, train_loader, val_loader, EPOCHS)
del train_loader, val_loader
gc.collect()

def nn_predict(X, chunk=262_144):
    return predict_with(net, X, chunk)

In [ ]:
# Test-set predictions -> saved for the main notebook's results table and plots
nn_test_pred = nn_predict(X_test).astype("float32")
print("NeuralNet", {
    "RMSE": round(float(rmse(y_test, nn_test_pred)), 4),
    "MAE": round(float(mean_absolute_error(y_test, nn_test_pred)), 4),
    "R2": round(float(r2_score(y_test, nn_test_pred)), 4),
})
np.save(MODEL_DIR / "nn_test_preds.npy", nn_test_pred)
np.save(MODEL_DIR / "nn_test_index.npy", X_test.index.to_numpy())   # lets the main notebook check alignment
torch.save(net.state_dict(), MODEL_DIR / "nn_state.pt")
print(f"Saved predictions and weights to {MODEL_DIR}/")

In [ ]:
def permutation_importance(predict, X, y, n_repeats=3, seed=42):
    rng = np.random.default_rng(seed)
    base_pred = predict(X)
    base_rmse, base_mae = rmse(y, base_pred), mean_absolute_error(y, base_pred)
    out = {"RMSE": {}, "MAE": {}}
    for col in X.columns:
        r_scores, m_scores = [], []
        for _ in range(n_repeats):
            Xp = X.copy()
            Xp[col] = rng.permutation(Xp[col].to_numpy())
            p = predict(Xp)
            r_scores.append(rmse(y, p))
            m_scores.append(mean_absolute_error(y, p))
        out["RMSE"][col] = np.mean(r_scores) - base_rmse
        out["MAE"][col] = np.mean(m_scores) - base_mae
    return pd.DataFrame(out)

# Same 200k-row sample and same shuffling seed as the main notebook's importance cell
X_imp = X_test.sample(200_000, random_state=0)
y_imp = y_test.loc[X_imp.index]
nn_imp = permutation_importance(nn_predict, X_imp, y_imp)
nn_imp.to_parquet(MODEL_DIR / "nn_importance.parquet")
print(nn_imp.sort_values("RMSE", ascending=False).round(4))